### Markov Chain in Continous Time

In [ ]:
import numpy as np

Q = np.array([[-1,1,0,0],
              [0,-1,1,0],
              [0,0,-1,1],
              [1,0,0,-1]])

eigenvalues,eigenvectors = np.linalg.eig(Q)
vecs = []
for i in range(len(eigenvalues)):
    val = eigenvalues[i]
    vec = eigenvectors[:, i] 
    vecs.append(vec)
    print(f"Eigenvalue {i+1}: {val}")
    print(f"Corresponding Eigenvector: {vec}\n")



Eigenvalue 1: (-2.0000000000000013+0j)
Corresponding Eigenvector: [ 0.5+0.j -0.5+0.j  0.5+0.j -0.5+0.j]

Eigenvalue 2: (-0.9999999999999993+0.9999999999999991j)
Corresponding Eigenvector: [ 5.00000000e-01-1.95522299e-16j  8.44999969e-17+5.00000000e-01j
 -5.00000000e-01+0.00000000e+00j  1.62566348e-16-5.00000000e-01j]

Eigenvalue 3: (-0.9999999999999993-0.9999999999999991j)
Corresponding Eigenvector: [ 5.00000000e-01+1.95522299e-16j  8.44999969e-17-5.00000000e-01j
 -5.00000000e-01-0.00000000e+00j  1.62566348e-16+5.00000000e-01j]

Eigenvalue 4: (-1.8732355726879631e-16+0j)
Corresponding Eigenvector: [0.5+0.j 0.5+0.j 0.5+0.j 0.5+0.j]



#### Part d) Solving Master Equations for initial condition

In [9]:
y0 = np.array([1/3,2/3,0,0])
c = np.linalg.solve(eigenvectors,y0)

for i in range(len(eigenvalues)):
    val = eigenvalues[i]
    vec = eigenvectors[:, i] 
    print(f"Eigenvalue {i+1}: {val}")
    print(f"Corresponding Eigenvector: {vec}")
    print(f'c_{i+1} = {c[i]}\n')


Eigenvalue 1: (-2.0000000000000013+0j)
Corresponding Eigenvector: [ 0.5+0.j -0.5+0.j  0.5+0.j -0.5+0.j]
c_1 = (-0.16666666666666644+0j)

Eigenvalue 2: (-0.9999999999999993+0.9999999999999991j)
Corresponding Eigenvector: [ 5.00000000e-01-1.95522299e-16j  8.44999969e-17+5.00000000e-01j
 -5.00000000e-01+0.00000000e+00j  1.62566348e-16-5.00000000e-01j]
c_2 = (0.16666666666666655-0.3333333333333332j)

Eigenvalue 3: (-0.9999999999999993-0.9999999999999991j)
Corresponding Eigenvector: [ 5.00000000e-01+1.95522299e-16j  8.44999969e-17-5.00000000e-01j
 -5.00000000e-01-0.00000000e+00j  1.62566348e-16+5.00000000e-01j]
c_3 = (0.1666666666666666+0.3333333333333332j)

Eigenvalue 4: (-1.8732355726879631e-16+0j)
Corresponding Eigenvector: [0.5+0.j 0.5+0.j 0.5+0.j 0.5+0.j]
c_4 = (0.5+7.622352818375916e-18j)



#### e) Simulation

In [13]:
import numpy as np
import plotly.graph_objects as go
def plot_2d(y:list,x:list = None, xlabel:str = None, ylabel:str = None,title:str = None,line_name:list[str] = None,xlims:tuple=None,ylims:tuple=None)-> None:
    """ 
    Do a simple 2D line chart (list of lines or other)
    """
    fig = go.Figure()
    

    if type(y[0]) == list or type(y[0]) == np.ndarray:
        for i in range(len(y)):
            if x is not None:
                fig.add_trace(go.Scatter(
                    x=x[i],
                    y=y[i],
                    mode='lines',
                    name=line_name[i]
                ))
            else:
                fig.add_trace(go.Scatter(
                    y=y[i],
                    mode='lines',
                    name=line_name[i]
                ))
    else:
        if x is not None:
            fig.add_trace(go.Scatter(
                x=x,
                y=y,
                mode='lines',
                name=line_name
            ))
        else:
            fig.add_trace(go.Scatter(
                y=y,
                mode='lines',
                name=line_name
            ))

    if title:
        fig.update_layout(
        title=title)
    if xlabel:
        fig.update_layout(
            xaxis_title = xlabel
        )
    if ylabel:
        fig.update_layout(
            yaxis_title = ylabel
        )

    if xlims is not None:
        fig.update_layout(xaxis=dict(range=[xlims[0], xlims[1]]))
    if ylims is not None:
        fig.update_layout(yaxis=dict(range=[ylims[0], ylims[1]]))
    # 4. Display the figure
    fig.show()
def continuous_chain(t_vec,l=1):
    """ 
    Simulate motion through the 4 state continuous markov chain
    """

    # Picking initial state
    state = 1 if np.random.rand() < 1/3 else 2

    # Initialization
    t = 0
    state_history = np.zeros(len(t_vec))
    index = 0

    while index < len(t_vec):

        # Finding time in that state and next time
        t_wait = np.random.exponential(l)
        t_next = t + t_wait

        # Filling in times before advancing states
        while index < len(t_vec) and t_vec[index] < t_next:
            state_history[index] = state
            index += 1

        # Advancing the state
        state = state % 4 + 1  
        t = t_next

    return state_history

def estimate_f(N,t_vec):
    """ 
    Estimate f(t) using the simulation of N Markov chains
    """

    count = np.zeros(len(t_vec))

    # Iterating through each Markov simulation
    for _ in range(N):
        states = continuous_chain(t_vec,l=1)
        count += (states == 1)

    return count/N

N_list  = [100,1000,10000,100000]
t_vec = np.linspace(0,5,200)
y_i = []

for N in N_list:
    f_estimate = estimate_f(N,t_vec)
    y_i.append(f_estimate)

f_analytical = lambda t: 1/4 -1/12*np.exp(-2*t) + np.exp(-t)*(np.cos(t)/6 - 1/3*np.sin(t))
analytical = f_analytical(t_vec)
y_i.append(analytical)
names = [f'{N} = 100',f'{N} = 1,000',f'{N} = 10,000',f'{N} = 100,000','Analytical']
plot_2d(y=y_i,x=[t_vec,t_vec,t_vec,t_vec,t_vec],title='Markov Chain Simulation',ylabel='f(t) = fraction in State 1',xlabel='Time(s)',line_name = names,xlims=(0,5),ylims=(0,1/2))